In [ ]:
from copy import deepcopy
from tqdm import tqdm

import sys
sys.path.insert(0, "../")

from src.db_drivers.vector_driver.connectors import MilvusConnector, DEFAULT_MILVUS_CONFIG
from src.db_drivers.vector_driver import VectorDBInstance, VectorDriverConfig, VectorDBConnectionConfig, VectorDriver
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig
from src.db_drivers.graph_driver.configs import DEFAULT_NEO4J_CONFIG

from src.pipelines.memorize import MemPipeline, MemPipelineConfig
from src.pipelines.memorize.MemPipeline import LLMExtractorConfig, LLMUpdatorConfig

from src.kg_model import KnowledgeGraphModel, GraphModelConfig, EmbedderModelConfig, EmbeddingsModelConfig

from src.pipelines.qa.kg_reasoning import KnowledgeGraphReasonerConfig
from src.pipelines.qa.kg_reasoning.weak_reasoner.knowledge_retriever import WaterCirclesSearchConfig, WaterCirclesTripletsRetriever
from src.pipelines.qa.kg_reasoning.weak_reasoner import QueryLLMParserConfig, KnowledgeComparatorConfig, \
    KnowledgeRetrieverConfig, QALLMGeneratorConfig, WeakKGReasonerConfig

from src.pipelines.qa import QAPipeline, QAPipelineConfig

from src.agents import AgentDriver, AgentDriverConfig
from src.agents.connectors.OpenAIConnector import DEEPSEEK_CONFIG, GPT4OMINI_CONFIG

In [ ]:
import pickle 
import hashlib

v = {'a': 2, 'b': 'c'}
d = pickle.dumps(v)
hashlib.sha1(d).hexdigest()

In [ ]:
'3f813fbd0510c6941817d2bb97d078384ac28baf'

In [ ]:
from src.db_drivers.vector_driver.connectors.MilvusConnector import MilvusConnector, DEFAULT_MILVUS_CONFIG

In [ ]:
DEFAULT_MILVUS_CONFIG.conn['port'] = 19534
DEFAULT_MILVUS_CONFIG.db_info['db'] = 'knowledgegraphdb'
DEFAULT_MILVUS_CONFIG.db_info['table'] = 'vectorized_triplets'
DEFAULT_MILVUS_CONFIG.params['load'] = False

In [ ]:
conn = MilvusConnector(DEFAULT_MILVUS_CONFIG)

In [ ]:
conn.close_connection()

In [ ]:
DEFAULT_MILVUS_CONFIG.params['load'] = False

In [ ]:
conn.open_connection()

In [ ]:
conn.client.list_indexes('vectorized_triplets')

In [ ]:
conn.client.release_collection('vectorized_triplets')

In [ ]:
conn.client.drop_index(collection_name='vectorized_triplets', index_name='embedding_index')

In [ ]:
import numpy as np

In [ ]:
query = VectorDBInstance(embedding=np.random.rand(1024))

In [ ]:
conn.retrieve([query])

In [ ]:
conn.count_items()

In [ ]:
from src.db_drivers.graph_driver.connectors.Neo4jConnector import Neo4jConnector, DEFAULT_NEO4J_CONFIG

In [ ]:
DEFAULT_NEO4J_CONFIG

In [ ]:
conn = Neo4jConnector()

In [ ]:
from src.utils.data_structs import TripletCreator, NodeCreator, RelationCreator, NodeType

In [ ]:
conn.create(
    triplets=[TripletCreator.create(
        start_node=NodeCreator.create(n_type='object', name='123'),
        relation=RelationCreator.create(r_type='simple', name='456'),
        end_node=NodeCreator.create(n_type='object', name='890'))]
)

In [ ]:
conn.read_by_name(name='123', object_type=NodeType.object, object='node')

In [ ]:
conn.open_connection()

In [ ]:
conn.close_connection()

In [ ]:
graph_config = GraphModelConfig(
    driver_config=GraphDriverConfig(
        db_vendor='neo4j',
        db_config=DEFAULT_NEO4J_CONFIG))

graph_config.driver_config.db_config.need_to_clear = True

embeddings_config = EmbeddingsModelConfig(
    nodesdb_driver_config=VectorDriverConfig(
        db_vendor='milvus', db_config=deepcopy(DEFAULT_MILVUS_CONFIG)),
    tripletsdb_driver_config=VectorDriverConfig(
        db_vendor='milvus',db_config=deepcopy(DEFAULT_MILVUS_CONFIG)),
    embedder_config=EmbedderModelConfig())

embeddings_config.nodesdb_driver_config.db_config.db_info['table'] = 'nodes_table'
embeddings_config.nodesdb_driver_config.db_config.params['vector_dim'] = 384
embeddings_config.nodesdb_driver_config.db_config.need_to_clear = True

embeddings_config.tripletsdb_driver_config.db_config.db_info['table'] = 'triplets_table'
embeddings_config.tripletsdb_driver_config.db_config.params['vector_dim'] = 384
embeddings_config.tripletsdb_driver_config.db_config.need_to_clear = True

In [ ]:
kg_model = KnowledgeGraphModel(graph_config, embeddings_config, verbose=True)

In [ ]:
AGENT_DRIVER = AgentDriverConfig(name='openai',agent_config=GPT4OMINI_CONFIG)

In [ ]:
mem_config = MemPipelineConfig(
    verbose=True,
    extractor_config=LLMExtractorConfig(
        verbose=True, lang='en',
        adriver_config=AGENT_DRIVER),
    updator_config=LLMUpdatorConfig(
        adriver_config=AGENT_DRIVER))

In [ ]:
memorize_pipeline = MemPipeline(kg_model, mem_config)

In [ ]:
messages = [
    "Mikhail Menshchikov is currently a second-year master's student at ITMO.",
    "Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'",
    "Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University",
    "Petrozavodsk State University is where Mikhail Menshchikov received his bachelor's degree.",
    "Mikhail Menshchikov studied at Petrozavodsk State University and received a bachelor's degree."]
properties = [dict() for _ in range(len(messages))]

In [ ]:
print(kg_model.graph_struct.count_items())
print(kg_model.embeddings_struct.count_items())

In [ ]:
for text, prop in tqdm(list(zip(messages, properties))):
    _, info = memorize_pipeline.remember(text=text, properties=prop)

In [ ]:
RETRIEVER_NAME = 'watercircles'
RETRIEVER_HYPERP = WaterCirclesSearchConfig(
    strict_filter = True, hyper_num = 15,
    episodic_num = 15, chain_triplets_num = 25,
    other_triplets_num = 6, do_text_pruning = False)

In [ ]:
LANGUAGE = 'en' # 'ru' | 'en' | 'auto'
# QA-pipeline configuration
QA_PIPELINE_CONFIG = QAPipelineConfig(
    verbose=True,
    reasoner_config=KnowledgeGraphReasonerConfig(
        verbose=True,
        reasoner_name='weak',
        reasoner_hyperparameters=WeakKGReasonerConfig(
            verbose=True,
            query_parser_config=QueryLLMParserConfig(lang=LANGUAGE, adriver_config=AGENT_DRIVER),
            knowledge_comparator_config=KnowledgeComparatorConfig(verbose=True),
            knowledge_retriever_config=KnowledgeRetrieverConfig(retriever_method=RETRIEVER_NAME,retriever_config=RETRIEVER_HYPERP),
            answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE, adriver_config=AGENT_DRIVER))))

In [ ]:
qa_pipeline = QAPipeline(kg_model=kg_model, config=QA_PIPELINE_CONFIG)

In [ ]:
answer, info = qa_pipeline.answer("What program is Mikhail Menshchikov studying for his master's degree?")
print(answer)

-------------------------

In [ ]:
config = VectorDriverConfig(db_vendor='milvus', db_config=VectorDBConnectionConfig(
        conn={'host': 'localhost', 'port': 19530, 'user': 'root', 'pass': 'Milvus'},
        db_info={'db': 'testing', 'table': 'testing'}, need_to_clear=True,
        params={'id_length': 32, 'vector_dim': 3, 'document_max_length': 51200, 
                'load': True, 'flush': False}))
conn = VectorDriver.select(config)

In [ ]:
instances = [
    VectorDBInstance(id='123', document='qwerty', embedding=[0.1, 0.2, 0.3], metadata={'k1': 'v1'}), 
    VectorDBInstance(id='456', document='ytrewq', embedding=[0.4, 0.5, 0.6], metadata={'k2': 'v2'})]

In [ ]:
from time import sleep

In [ ]:
conn.clear()
conn.create(instances)
sleep(1)
conn.item_exist('123')

In [ ]:
conn.item_exist('123')

In [ ]:
conn.item_exist('123')

In [ ]:
example = VectorDBInstance(id='1'*32, document='hello world', embedding=np.random.rand(1024))
example1_1 = VectorDBInstance(id='1'*32, document='hello world 4', embedding=np.random.rand(1024))
example2 = VectorDBInstance(id='2'*32, document='hello world 2', embedding=np.random.rand(1024))
example3 = VectorDBInstance(id='3'*32, document='hello world 3', embedding=np.random.rand(1024))

In [ ]:
conn.open_connection()
conn.is_open()

In [ ]:
conn.close_connection()
conn.is_open()

In [ ]:
conn.create([example, example2, example3])

In [ ]:
conn.item_exist(example3.id)

In [ ]:
conn.retrieve([example], n_results=2)